# Preparation des Donnees pour le Dashboard

Ce notebook prepare et exporte tous les fichiers necessaires au dashboard Plotly Dash.

**Prerequis** : scripts `04_baseline_model.py` et `05_advanced_model.py` deja executes.

Fichiers exportes dans `dashboard/data/` :
- `dashboard_main.parquet` : predictions + reel + erreurs
- `dashboard_daily.parquet` : agregation journaliere
- `dashboard_hourly_profile.parquet` : profil horaire moyen
- `dashboard_hourly_monthly.parquet` : profil horaire x mois
- `dashboard_robustness.parquet` : metriques par segment
- `dashboard_metrics.json` : KPIs globaux


In [13]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join('..', 'src'))
from utils import mae, rmse, smape, get_us_federal_holidays

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
DASHBOARD_DIR = os.path.join('..', 'dashboard', 'data')
os.makedirs(DASHBOARD_DIR, exist_ok=True)

print('Setup OK - dossier dashboard/data/ pret')


Setup OK - dossier dashboard/data/ pret


## 1. Chargement de toutes les sources


In [37]:
# Donnees horaires brutes
hourly = pd.read_parquet(os.path.join(PROCESSED_DIR, 'trips_hourly.parquet'))
hourly['datetime'] = pd.to_datetime(hourly['datetime'])

# Features engineered
features = pd.read_parquet(os.path.join(PROCESSED_DIR, 'features.parquet'))
features['datetime'] = pd.to_datetime(features['datetime'])

# Predictions baseline
baseline_preds = pd.read_parquet(os.path.join(PROCESSED_DIR, 'baseline_predictions.parquet'))
baseline_preds['datetime'] = pd.to_datetime(baseline_preds['datetime'])

# Predictions XGBoost
xgb_preds = pd.read_parquet(os.path.join(PROCESSED_DIR, 'xgb_predictions.parquet'))
xgb_preds['datetime'] = pd.to_datetime(xgb_preds['datetime'])

print(f'Horaire  : {len(hourly):,} lignes ({hourly["datetime"].min().date()} -> {hourly["datetime"].max().date()})')
print(f'Features : {len(features):,} lignes')
print(f'Baseline : {len(baseline_preds):,} lignes (jeu de test)')
print(f'XGBoost  : {len(xgb_preds):,} lignes (jeu de test)')


Horaire  : 17,544 lignes (2024-01-01 -> 2025-12-31)
Features : 17,376 lignes
Baseline : 2,231 lignes (jeu de test)
XGBoost  : 10,247 lignes (jeu de test)


## 2. Table principale - toutes predictions reunies


In [38]:
# Merge : XGBoost + baseline + features meteo et calendrier
main = xgb_preds.merge(
    baseline_preds[['datetime', 'pred_naive', 'pred_wwa']],
    on='datetime', how='left'
)

# Colonnes meteo et calendrier disponibles dans features
feat_cols = ['datetime', 'hour', 'day_of_week', 'month', 'is_weekend']
for col in ['is_holiday', 'temperature', 'precipitation', 'is_rain', 'is_snow']:
    if col in features.columns:
        feat_cols.append(col)

main = main.merge(features[feat_cols], on='datetime', how='left')

# Colonnes temporelles
main['date']  = main['datetime'].dt.date
main['hour']  = main['datetime'].dt.hour
main['month'] = main['datetime'].dt.month
main['dow']   = main['datetime'].dt.dayofweek
main['week']  = main['datetime'].dt.isocalendar().week.astype(int)

# Erreurs absolues
main['err_naive'] = (main['total_rides'] - main['pred_naive'].fillna(0)).abs()
main['err_wwa']   = (main['total_rides'] - main['pred_wwa'].fillna(0)).abs()
main['err_xgb']   = (main['total_rides'] - main['pred_xgb']).abs()

# Erreur relative XGBoost
denom = main['total_rides'].abs() + main['pred_xgb'].abs() + 1e-8
main['smape_xgb'] = 200 * main['err_xgb'] / denom

# Flag pics (top 10%)
peak_threshold = main['total_rides'].quantile(0.90)
main['is_peak'] = (main['total_rides'] >= peak_threshold).astype(int)

# MAE glissante sur 24h
main = main.sort_values('datetime').reset_index(drop=True)
main['mae_rolling_24h'] = main['err_xgb'].rolling(24, min_periods=1).mean()

print(f'Table principale : {main.shape}')
print(f'Colonnes         : {list(main.columns)}')
display(main.head(3))


Table principale : (10247, 24)
Colonnes         : ['datetime', 'total_rides', 'pred_xgb', 'residual', 'pred_naive', 'pred_wwa', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday', 'temperature', 'precipitation', 'is_rain', 'is_snow', 'date', 'dow', 'week', 'err_naive', 'err_wwa', 'err_xgb', 'smape_xgb', 'is_peak', 'mae_rolling_24h']


,datetime,total_rides,pred_xgb,residual,pred_naive,pred_wwa,hour,day_of_week,month,is_weekend,...,is_snow,date,dow,week,err_naive,err_wwa,err_xgb,smape_xgb,is_peak,mae_rolling_24h
0,2024-10-31 01:00:00,93,83.752266,9.247734,NaN,NaN,1,3,10,0,...,0,2024-10-31,3,44,93.0,93.0,9.247734,10.464063,0,9.247734
1,2024-10-31 02:00:00,15,58.486752,-43.486752,NaN,NaN,2,3,10,0,...,0,2024-10-31,3,44,15.0,15.0,43.486752,118.352630,0,26.367243
2,2024-10-31 03:00:00,31,45.958794,-14.958794,NaN,NaN,3,3,10,0,...,0,2024-10-31,3,44,31.0,31.0,14.958794,38.874813,0,22.564426


## 3. Agregation journaliere


In [39]:
daily_tmp = hourly.copy()
daily_tmp['date']  = daily_tmp['datetime'].dt.date
daily_tmp['month'] = daily_tmp['datetime'].dt.month
daily_tmp['dow']   = daily_tmp['datetime'].dt.dayofweek
daily_tmp['week']  = daily_tmp['datetime'].dt.isocalendar().week.astype(int)

daily = daily_tmp.groupby('date').agg(
    total_rides  = ('total_rides',  'sum'),
    member_rides = ('member_rides', 'sum'),
    casual_rides = ('casual_rides', 'sum'),
    month        = ('month', 'first'),
    dow          = ('dow',   'first'),
    week         = ('week',  'first'),
).reset_index()
daily['date'] = pd.to_datetime(daily['date'])

# Jours feries
years = daily['date'].dt.year.unique().tolist()
holidays = get_us_federal_holidays(years)
daily['is_holiday'] = daily['date'].isin(holidays).astype(int)

# Predictions agreges sur la periode de test
test_daily = main.groupby('date').agg(
    pred_naive = ('pred_naive', 'sum'),
    pred_wwa   = ('pred_wwa',   'sum'),
    pred_xgb   = ('pred_xgb',   'sum'),
).reset_index()
test_daily['date'] = pd.to_datetime(test_daily['date'])
daily = daily.merge(test_daily, on='date', how='left')

# Variation j/j
daily = daily.sort_values('date').reset_index(drop=True)
daily['pct_change'] = daily['total_rides'].pct_change() * 100

print(f'Agregat journalier : {daily.shape}')
display(daily.tail(5))


Agregat journalier : (731, 12)


,date,total_rides,member_rides,casual_rides,month,dow,week,is_holiday,pred_naive,pred_wwa,pred_xgb,pct_change
726,2025-12-27,5304,3708,1596,12,5,52,0,9677.0,10808.4,11842.752930,26.859603
727,2025-12-28,5296,3831,1465,12,6,52,0,7525.0,7489.3,10531.127930,-0.150830
728,2025-12-29,7067,5353,1714,12,0,1,0,8176.0,9081.0,9672.767578,33.440332
729,2025-12-30,6241,4926,1315,12,1,1,0,7112.0,9307.4,9934.786133,-11.688128
730,2025-12-31,6939,5360,1579,12,2,1,0,6092.0,10455.7,11998.514648,11.184105


## 4. Profils horaires moyens


In [40]:
hourly_work = hourly.copy()
hourly_work['hour']       = hourly_work['datetime'].dt.hour
hourly_work['dow']        = hourly_work['datetime'].dt.dayofweek
hourly_work['month']      = hourly_work['datetime'].dt.month
hourly_work['is_weekend'] = (hourly_work['dow'] >= 5).astype(int)
hourly_work['date_n']     = hourly_work['datetime'].dt.normalize()

holidays_all = get_us_federal_holidays(hourly_work['datetime'].dt.year.unique().tolist())
hourly_work['is_holiday'] = hourly_work['date_n'].isin(holidays_all).astype(int)

def classify_day(row):
    if row['is_holiday']:   return 'Jour ferie'
    elif row['is_weekend']: return 'Week-end'
    else:                   return 'Semaine'

hourly_work['day_type'] = hourly_work.apply(classify_day, axis=1)

# Profil heure x type de jour
profile = hourly_work.groupby(['hour', 'day_type'])['total_rides'].agg(
    ['mean', 'std', 'median', 'count']
).reset_index()
profile.columns = ['hour', 'day_type', 'mean_rides', 'std_rides', 'median_rides', 'n']
import numpy as np
profile['ci_low']  = profile['mean_rides'] - 1.96 * profile['std_rides'] / np.sqrt(profile['n'])
profile['ci_high'] = profile['mean_rides'] + 1.96 * profile['std_rides'] / np.sqrt(profile['n'])

# Profil heure x mois
profile_monthly = hourly_work.groupby(['hour', 'month'])['total_rides'].mean().reset_index()
profile_monthly.columns = ['hour', 'month', 'mean_rides']

print(f'Profil horaire (heure x type jour) : {profile.shape}')
print(f'Profil mensuel (heure x mois)      : {profile_monthly.shape}')
display(profile.head(6))


Profil horaire (heure x type jour) : (72, 8)
Profil mensuel (heure x mois)      : (288, 3)


,hour,day_type,mean_rides,std_rides,median_rides,n,ci_low,ci_high
0,0,Jour ferie,218.000000,120.325353,198.0,22,167.719235,268.280765
1,0,Semaine,132.686627,64.635684,128.0,501,127.026716,138.346537
2,0,Week-end,387.899038,153.274199,408.5,208,367.068838,408.729239
3,1,Jour ferie,138.954545,113.647524,127.0,22,91.464268,186.444823
4,1,Semaine,62.742515,34.928378,56.0,501,59.683965,65.801065
5,1,Week-end,267.125000,110.834532,282.5,208,252.062416,282.187584


## 5. Metriques de robustesse par segment


In [41]:
def metrics_segment(df_seg, pred_col='pred_xgb', y_col='total_rides', label=''):
    """Calcule MAE, sMAPE et biais pour un segment."""
    valid = df_seg[[y_col, pred_col, 'pred_naive']].dropna()
    if len(valid) < 5:
        return None
    y = valid[y_col].values
    p = valid[pred_col].values
    n = valid['pred_naive'].fillna(0).values
    return {
        'segment':     label,
        'n':           len(valid),
        'mean_demand': round(float(y.mean()), 1),
        'MAE_naive':   round(float(mae(y, n)), 1),
        'MAE_xgb':     round(float(mae(y, p)), 1),
        'sMAPE_xgb':   round(float(smape(y, p)), 1),
        'bias_xgb':    round(float((p - y).mean()), 1),
    }

segments = []

# Par type de jour
holiday_col = main.get('is_holiday', pd.Series(0, index=main.index))
for label, mask in [
    ('Semaine normale', (main['is_weekend'] == 0) & (holiday_col == 0)),
    ('Week-end',         main['is_weekend'] == 1),
    ('Jour ferie',       holiday_col == 1),
]:
    r = metrics_segment(main[mask], label=label)
    if r: segments.append(r)

# Par plage horaire
for label, hours in [
    ('Nuit (0-6h)',       range(0, 7)),
    ('Rush matin (7-9h)', range(7, 10)),
    ('Journee (10-16h)',  range(10, 17)),
    ('Rush soir (17-19h)',range(17, 20)),
    ('Soiree (20-23h)',   range(20, 24)),
]:
    r = metrics_segment(main[main['hour'].isin(hours)], label=label)
    if r: segments.append(r)

# Pics vs creux
for label, mask in [
    ('Pic (top 10%)',    main['is_peak'] == 1),
    ('Normal (mid 50%)', (main['total_rides'] >= main['total_rides'].quantile(0.25)) &
                         (main['total_rides'] <  main['total_rides'].quantile(0.75))),
    ('Creux (bot 10%)',  main['total_rides'] <= main['total_rides'].quantile(0.10)),
]:
    r = metrics_segment(main[mask], label=label)
    if r: segments.append(r)

# Meteo (si disponible)
if 'is_rain' in main.columns:
    for label, mask in [
        ('Meteo sec',  main['is_rain'].fillna(0) == 0),
        ('Meteo pluie', main['is_rain'].fillna(0) == 1),
    ]:
        r = metrics_segment(main[mask], label=label)
        if r: segments.append(r)

robustness_df = pd.DataFrame(segments)
robustness_df['gain_pct'] = (
    100 * (robustness_df['MAE_naive'] - robustness_df['MAE_xgb'])
    / robustness_df['MAE_naive']
).round(1)

print(f'Segments analyses : {len(robustness_df)}')
display(robustness_df)


Segments analyses : 13


,segment,n,mean_demand,MAE_naive,MAE_xgb,sMAPE_xgb,bias_xgb,gain_pct
0,Semaine normale,1511,640.3,119.6,115.1,29.6,29.6,3.8
1,Week-end,624,616.8,172.2,137.8,31.3,78.3,20.0
2,Jour ferie,96,352.0,373.2,276.3,68.8,259.7,26.0
3,Nuit (0-6h),650,111.5,35.0,43.8,50.5,35.3,-25.1
4,Rush matin (7-9h),279,769.7,211.3,177.9,29.6,78.2,15.8
5,Journee (10-16h),651,913.1,191.9,152.1,19.3,58.1,20.7
6,Rush soir (17-19h),279,1190.0,278.2,255.5,26.9,18.3,8.2
7,Soiree (20-23h),372,463.5,106.8,102.0,26.1,82.8,4.5
8,Pic (top 10%),133,1965.9,274.4,334.8,18.3,-329.2,-22.0
9,Normal (mid 50%),1222,602.3,164.3,140.2,23.5,107.8,14.7


## 6. KPIs globaux pour les cards du dashboard


In [42]:
y_true = main['total_rides'].values
p_xgb  = main['pred_xgb'].values
p_naif = main['pred_naive'].fillna(0).values

mae_xgb   = float(mae(y_true, p_xgb))
mae_naif  = float(mae(y_true, p_naif))

kpis = {
    # Informations generales
    'total_rides_all':       int(hourly['total_rides'].sum()),
    'total_rides_test':      int(main['total_rides'].sum()),
    'n_hours_total':         int(len(hourly)),
    'n_hours_test':          int(len(main)),
    'period_start':          str(hourly['datetime'].min().date()),
    'period_end':            str(hourly['datetime'].max().date()),
    'test_start':            str(main['datetime'].min().date()),
    'test_end':              str(main['datetime'].max().date()),
    # Performance XGBoost
    'xgb_mae':               round(mae_xgb, 2),
    'xgb_rmse':              round(float(rmse(y_true, p_xgb)), 2),
    'xgb_smape':             round(float(smape(y_true, p_xgb)), 2),
    # Performance Baseline
    'naive_mae':             round(mae_naif, 2),
    'naive_smape':           round(float(smape(y_true, p_naif)), 2),
    # Gain relatif
    'improvement_mae_pct':   round(100 * (mae_naif - mae_xgb) / mae_naif, 1),
    # Statistiques demande
    'avg_hourly_demand':     round(float(hourly['total_rides'].mean()), 1),
    'peak_demand':           int(hourly['total_rides'].max()),
    'peak_hour_threshold':   round(float(main['total_rides'].quantile(0.90)), 1),
    # Part member/casual
    'pct_member':            round(100 * int(hourly['member_rides'].sum()) / int(hourly['total_rides'].sum()), 1),
    'pct_casual':            round(100 * int(hourly['casual_rides'].sum()) / int(hourly['total_rides'].sum()), 1),
}

print('KPIs calcules :')
for k, v in kpis.items():
    print(f'  {k:<35} : {v}')


KPIs calcules :
  total_rides_all                     : 12776970
  total_rides_test                    : 7623121
  n_hours_total                       : 17544
  n_hours_test                        : 10247
  period_start                        : 2024-01-01
  period_end                          : 2025-12-31
  test_start                          : 2024-10-31
  test_end                            : 2025-12-31
  xgb_mae                             : 142.17
  xgb_rmse                            : 221.07
  xgb_smape                           : 28.16
  naive_mae                           : 640.29
  naive_smape                         : 162.76
  improvement_mae_pct                 : 77.8
  avg_hourly_demand                   : 728.3
  peak_demand                         : 3180
  peak_hour_threshold                 : 1661.4
  pct_member                          : 68.8
  pct_casual                          : 31.2


In [43]:
print("y_true min/max:", y_true.min(), y_true.max())
print("p_naif min/max:", p_naif.min(), p_naif.max())
print("mae_naif:", mae_naif)
print("nb zeros y_true:", (y_true == 0).sum())
print("nb rows:", len(main))

y_true min/max: 0 3180
p_naif min/max: 0.0 2827.0
mae_naif: 640.287791548746
nb zeros y_true: 1
nb rows: 10247


In [44]:
print(hourly.groupby(hourly["datetime"].dt.year)["total_rides"].sum())
print(hourly["total_rides"].describe())
print(hourly["total_rides"].value_counts().head(10))

datetime
2024    6114359
2025    6662611
Name: total_rides, dtype: int32
count    17544.000000
mean       728.281464
std        609.749201
min          0.000000
25%        182.000000
50%        625.000000
75%       1105.000000
max       3180.000000
Name: total_rides, dtype: float64
total_rides
31    64
27    64
33    56
21    54
45    54
23    54
44    52
39    52
26    51
30    50
Name: count, dtype: int64


## 7. Export de tous les fichiers


In [45]:
# Export Parquet
exports = [
    (main,            'dashboard_main.parquet'),
    (daily,           'dashboard_daily.parquet'),
    (profile,         'dashboard_hourly_profile.parquet'),
    (profile_monthly, 'dashboard_hourly_monthly.parquet'),
    (robustness_df,   'dashboard_robustness.parquet'),
]

for df_exp, filename in exports:
    path = os.path.join(DASHBOARD_DIR, filename)
    df_exp.to_parquet(path, index=False)
    size_kb = os.path.getsize(path) // 1024
    print(f'  OK {filename:<45} ({len(df_exp):>6,} lignes, {size_kb} KB)')

# Export JSON pour les KPIs
kpi_path = os.path.join(DASHBOARD_DIR, 'dashboard_metrics.json')
with open(kpi_path, 'w', encoding='utf-8') as f:
    json.dump(kpis, f, indent=2, ensure_ascii=True)
print(f'  OK dashboard_metrics.json')

print(f'\nTous les fichiers exportes dans : {DASHBOARD_DIR}/')
print(f'Total fichiers : {len(exports) + 1}')


  OK dashboard_main.parquet                        (10,247 lignes, 607 KB)
  OK dashboard_daily.parquet                       (   731 lignes, 34 KB)
  OK dashboard_hourly_profile.parquet              (    72 lignes, 7 KB)
  OK dashboard_hourly_monthly.parquet              (   288 lignes, 4 KB)
  OK dashboard_robustness.parquet                  (    13 lignes, 5 KB)
  OK dashboard_metrics.json

Tous les fichiers exportes dans : ..\dashboard\data/
Total fichiers : 6


## 8. Verification finale


In [46]:
print('Verification des fichiers exportes :')
all_files = [
    'dashboard_main.parquet',
    'dashboard_daily.parquet',
    'dashboard_hourly_profile.parquet',
    'dashboard_hourly_monthly.parquet',
    'dashboard_robustness.parquet',
    'dashboard_metrics.json',
]
for fname in all_files:
    path = os.path.join(DASHBOARD_DIR, fname)
    if os.path.exists(path):
        size_kb = os.path.getsize(path) // 1024
        print(f'  OK {fname:<45} ({size_kb} KB)')
    else:
        print(f'  MANQUANT : {fname}')

print()
print('Vous pouvez maintenant lancer le dashboard :')
print('  python dashboard/app.py')
print('  -> http://127.0.0.1:8050')


Verification des fichiers exportes :
  OK dashboard_main.parquet                        (607 KB)
  OK dashboard_daily.parquet                       (34 KB)
  OK dashboard_hourly_profile.parquet              (7 KB)
  OK dashboard_hourly_monthly.parquet              (4 KB)
  OK dashboard_robustness.parquet                  (5 KB)
  OK dashboard_metrics.json                        (0 KB)

Vous pouvez maintenant lancer le dashboard :
  python dashboard/app.py
  -> http://127.0.0.1:8050
